# 수도권제1순환고속도로 속도 예측 — Google Colaboratory
> ⚠️ Google Colaboratory 런타임 환경 기준으로 작성되어 있습니다.

## 런타임 설정

In [ ]:
import subprocess, os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

subprocess.run("apt-get install -y htop nvtop", shell=True, check=True, capture_output=True)
subprocess.run("apt-get install -y fonts-nanum*", shell=True, check=True, capture_output=True)
subprocess.run("pip install xlsxwriter fastexcel openpyxl tensordict torchao torchdata pyzmq netifaces", shell=True, check=True, capture_output=True)
subprocess.run("pip install pyarrow[flight,dataset] --extra-index-url https://pypi.nvidia.com/", shell=True, check=True, capture_output=True)
if os.path.exists("/root/.cache/matplotlib"):
  subprocess.run("rm -rf /root/.cache/matplotlib", shell=True, check=True)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
print("✅ Setup complete. Please restart the runtime for font changes to take effect.")

In [ ]:
import subprocess

subprocess.run("pip install transformer-engine[pytorch]", shell=True, check=True, capture_output=True)
print("✅ NVIDIA Transformer Engine for PyTorch is successfully Instelled.")
subprocess.run("pip install torchscale", shell=True, check=True, capture_output=True)
print("✅ TorchScale is successfully Instelled.")

## 모델 및 PyTorch 로드

In [ ]:
import os
import google.colab.drive as gd

gd.mount("/content/drive", force_remount=True)
os.chdir(os.path.join(os.getcwd(), "drive/My Drive/Colab Notebooks/수도권제1순환고속도로 통행 속도 추세 분석하기 (vNext)/"))

import stformer_torch
import torch

device = stformer_torch.toolkit.capability.get_device()
print(f"✅ Torch initialized with {device.type.upper()}.")

## RAW 데이터 로드

In [ ]:
import polars as pl
import openpyxl

xlsx_name = "수도권제1순환고속도로.xlsx"
workbook = openpyxl.load_workbook(xlsx_name, read_only=True, data_only=True)
sheet_names = workbook.sheetnames
raw_data = {
    sheet_name: pl.read_excel(xlsx_name, sheet_name=sheet_name).lazy()
    for sheet_name in sheet_names
}

## 피처, 라벨 인코딩

In [ ]:
import re
from typing import Union
import numpy as np

def parse(name: str) -> tuple[int, str]:
    m = re.search(r"^\d+", name)
    month = int(m.group()) if m else None
    daytype = "평일" if "평일" in name else "주말·공휴일"
    return month, daytype

def cleanse(
    df: Union[pl.DataFrame, pl.LazyFrame],
    month: int,
    daytype: str
) -> Union[pl.DataFrame, pl.LazyFrame]:

    if isinstance(df, pl.DataFrame):
        hour_cols = [c for c in HOURS if c in df.columns]
    elif isinstance(df, pl.LazyFrame):
        hour_cols = [c for c in HOURS if c in df.collect_schema().names()]
    else:
        raise TypeError("df must be a Polars DataFrame or LazyFrame.")
    return (
        df.lazy()
        .with_columns(
            pl.col("구간").cast(pl.String),
            pl.col("방향").cast(pl.String),
        )
        # 구간 문자열 정상화 + 하이픈이 없는 행 제거
        .filter(pl.col("구간").is_not_null() & pl.col("구간").str.contains("-", literal=True))
        .with_columns([pl.col(c).cast(pl.Float32, strict=False) for c in hour_cols])
        .with_columns(
            pl.col("구간").str.replace_all(r"\s+", "").alias("구간"),
            pl.col("방향").str.strip_chars().alias("방향"),
            # 양끝 정렬해서 동일 구간을 한 ID로 (예: A-B, B-A → A-B)
            pl.col("구간").str.split("-").list.sort().list.join("-").alias("구간ID"),
        )
        # wide → long
        .unpivot(index=["구간","구간ID","방향"], on=hour_cols,
              variable_name="시간문자", value_name="속도")  # DataFrame.melt 문서 참고
        .with_columns(
            pl.lit(month).alias("월"),
            pl.lit(DAY2ID[daytype]).alias("요일타입_id"),
            pl.when(pl.col("방향")=="상행").then(0)
              .when(pl.col("방향")=="하행").then(1)
              .otherwise(None).alias("방향_id"),
            pl.col("시간문자").str.replace_all("시","").cast(pl.Int32).alias("시간"),
            pl.col("속도").cast(pl.Float32, strict=False),
        )
        .select(["월","요일타입_id","방향_id","구간ID","시간","속도"])
        .filter(pl.col("방향_id").is_not_null())
        .group_by(["월","요일타입_id","방향_id","구간ID","시간"])
        .agg(pl.col("속도").mean().alias("속도"))
        .collect()
    )

def nanmean_grid(grid: np.ndarray) -> np.ndarray:
    col_mean = np.nanmean(grid, axis=0, keepdims=True)
    col_mean = np.where(np.isnan(col_mean), 0.0, col_mean)
    grid = np.where(np.isnan(grid), col_mean, grid)
    return grid

In [ ]:
HOURS   = [f"{h:02d}시" for h in range(24)]
DAY2ID  = {"평일": 0, "주말·공휴일": 1}
DIR2ID  = {"상행": 0, "하행": 1}
DIR_ID2NAME = {v: k for k, v in DIR2ID.items()}
long_parts = []

for name in sheet_names:
    m, daytype = parse(name)
    long_parts.append(cleanse(raw_data[name], m, daytype))
df = pl.concat(long_parts, how="vertical", rechunk=True)

segments = df.select("구간ID").unique().sort("구간ID").get_column("구간ID").to_list()
seg2idx = {s: i for i, s in enumerate(segments)}
S, T = len(segments), 24
train_data_3d = {}

for (m, d_id, dir_id), g in df.group_by(["월","요일타입_id","방향_id"]):
    # 빈 그리드 생성
    grid = np.full((T, S, 1), np.nan, dtype=np.float32)
    # 현재 그룹에서 값 채우기
    for (t, seg, v) in g.select(["시간","구간ID","속도"]).iter_rows():
        s = seg2idx[seg]
        if v is not None and not (np.isnan(v)):
            grid[int(t), int(s), 0] = float(v)
    grid = nanmean_grid(grid)
    y = torch.from_numpy(grid)
    key = (int(m), int(d_id), int(dir_id))
    train_data_3d[key] = y

##모델 생성

In [ ]:
config = (
    stformer_torch.workflow.NetConfig(
        device="cuda", microbatch=128, d_model=1152, nhead=12, depth=12, mlp_ratio=3.0,
        dropout=0.0, drop_path_max=0.1, drop_path_schedule="linear",
        norm_type="rmsnorm", simultaneous_retention=True, long_transformer_window = 512,
        micro_grad_checkpoint=False, macro_grad_checkpoint=False,
        compile_subnet=True, compile_net=True, compile_mode="max-autotune",
        pad_to_patch_multiple=True, feat_use_cube=False, feat_use_square=False,
        out_use_square=False, out_use_cube=True, out_grid_3d=(T, S, 1), out_patch_3d=(1,1,1),
        feat_patch_side_2d=1, feat_grid_side_2d=1,
        feat_patch_size_1d=1, out_patch_size_1d=1
    )
    if device.type == 'cuda' else
    stformer_torch.workflow.NetConfig(
        device="cpu", microbatch=512,
        d_model=144, nhead=6, depth=12, mlp_ratio=3.0,
        norm_type="layernorm", simultaneous_retention=True, long_transformer_window = 256,
        dropout=0.10, drop_path_max=0.0, drop_path_schedule="linear",
        micro_grad_checkpoint=False, macro_grad_checkpoint=False,
        compile_subnet=True, compile_net=True, compile_mode="max-autotune-no-cudagraphs",
        pad_to_patch_multiple=True, feat_use_cube=False, feat_use_square=False,
        out_use_square=False, out_use_cube=True, out_grid_3d=(T, S, 1), out_patch_3d=(1,1,1),
        feat_patch_side_2d=1, feat_grid_side_2d=1,
        feat_patch_size_1d=1, out_patch_size_1d=1
    )
)

In [ ]:
in_dim = 3
out_shape = (T, S, 1)
model = stformer_torch.workflow.management.new_model(config=config, in_dim=in_dim, out_shape=out_shape)

##모델 학습

In [ ]:
parameters = {
    "epochs": 200,
    "batch_size": 256,
    "base_lr": 0.0005,
    "weight_decay": 0.0001,
    "warmup_ratio": 0.06,
    "val_frac": 0.2,
    "prefetch_factor": 4,
    "grad_accum_steps": 8
} if device.type == 'cuda' else {
    "epochs": 10,
    "batch_size": 512,
    "base_lr": 0.0005,
    "weight_decay": 0.0001,
    "warmup_ratio": 0.06,
    "val_frac": 0.2,
    "prefetch_factor": 8,
    "grad_accum_steps": 2
}

In [ ]:
model = stformer_torch.workflow.operation.train(model=model, data=train_data_3d, **parameters)

##추론 데이터를 저장할 템플릿 생성

In [ ]:
import copy

template = copy.deepcopy(raw_data)
time_columns = [col for col in next(iter(template.values())).collect_schema().names() if '시' in col]
template = {
    sheet_name: lf.with_columns(pl.lit(0).alias(col) for col in time_columns)
    for sheet_name, lf in template.items()
}

##모델 추론

In [ ]:
infer_data = {k: None for k in train_data_3d.keys()}
pred_dict = stformer_torch.workflow.operation.predict(model=model, data=infer_data, batch_size=512, prefetch_factor=4)

## 추론 데이터 저장

In [ ]:
from typing import Tuple, Sequence, Dict

def _is_hour_col(name: str) -> bool:
    s = str(name)
    return len(s)==3 and s.endswith("시") and s[:2].isdigit()

def get_schema(raw_data: Dict[str, pl.LazyFrame]) -> Tuple[Sequence[str], Sequence[str]]:
    """원본 시트에서 왼쪽 전체 컬럼(schema_left)과 시간열 순서(schema_right) 검출."""
    first_sheet = next(iter(raw_data.values()))
    cols = first_sheet.collect_schema().names()
    idx_first_hour = min([i for i,c in enumerate(cols) if _is_hour_col(c)], default=len(cols))
    schema_left = list(cols[:idx_first_hour]) if idx_first_hour>0 else ["구간","방향"]
    schema_right = HOURS  # 저장은 00~23 고정이 안전
    return (schema_left, schema_right)

# === 2) 구간/방향 복원 ===
def categorize_poi(seg_id: str, dir_id: int, delim: str = "-") -> str:
    """정규화된 'A-B'(seg_id) + 방향 id(상=0/하=1) → 원래 '구간'(A-B 또는 B-A)"""
    a, b = seg_id.split(delim, 1)
    return f"{a}{delim}{b}" if int(dir_id) == 0 else f"{b}{delim}{a}"

# === 3) (T,S,1) grid → long DF ===
def to_long_polars(key: Tuple[int,int,int], grid: torch.Tensor, seg_ids: Sequence[str]) -> pl.DataFrame:
    m, day_id, dir_id = map(int, key)
    T, S, C = grid.shape
    assert C == 1
    g = grid.detach().to("cpu").float().numpy()
    recs = [(m, day_id, dir_id, seg_ids[s], t, float(g[t, s, 0]))
            for t in range(T) for s in range(S)]
    # ⬇️ 행방향 명시
    return pl.DataFrame(recs,
                        schema=["월","요일타입_id","방향_id","구간ID","시간","속도"],
                        orient="row")

# === 4) long → wide(pivot) ===
def to_wide_polars(long_df: pl.DataFrame, index_cols: Sequence[str]) -> pl.DataFrame:
    # pivot은 eager에서만 가능
    w = (long_df
         # ⬇️ columns -> on 으로 교체
         .pivot(values="속도", index=index_cols, on="시간", aggregate_function="mean"))  # new API

    # pivot 뒤 시간 컬럼은 문자열로 존재하는 경우가 많음 -> index_cols 제외가 시간열
    time_cols = [c for c in w.columns if c not in index_cols]
    have = {int(c) for c in time_cols if str(c).isdigit()}

    # 누락된 시간열 보강 (열 이름은 반드시 문자열)
    for h in range(24):
        if h not in have:
            w = w.with_columns(pl.lit(None).alias(str(h)))

    # 최종 순서: index_cols + 0..23 (문자열) -> '00시'..'23시'로 rename
    w = w.select(list(index_cols) + [str(h) for h in range(24)])
    return w.rename({str(h): f"{h:02d}시" for h in range(24)})

# === 5) 템플릿에 예측 덮어쓰기 ===
def to_polars(template_lf: pl.LazyFrame,
              schema_left: Sequence[str],
              schema_right: Sequence[str],
              pred_long_for_sheet: pl.DataFrame) -> pl.DataFrame:
    temp = template_lf.collect()
    assert "구간" in temp.columns and "방향" in temp.columns, "템플릿에 '구간'/'방향' 필요"

    temp = (temp
            .with_columns([
                pl.col("구간").cast(pl.String).str.split("-").list.sort().list.join("-").alias("구간ID"),
                # ⬇️ 반환 dtype 명시
                pl.col("방향").map_elements(lambda s: DIR2ID.get(s, 1), return_dtype=pl.Int32).alias("방향_id"),
            ]))

    # long 예측 → wide
    wide_pred = to_wide_polars(pred_long_for_sheet, index_cols=["구간ID","방향_id"])

    # 기존 시간열 목록
    time_cols_in_template = [c for c in temp.columns if _is_hour_col(c)]

    # join & 시간열 교체
    joined = temp.join(wide_pred, on=["구간ID","방향_id"], how="left", suffix="_pred")
    final = joined
    for h in schema_right:            # "00시".."23시"
        pred_col = f"{h}_pred"
        if pred_col in final.columns:
            final = final.with_columns(
                pl.when(pl.col(pred_col).is_not_null())
                  .then(pl.col(pred_col))
                  .otherwise(pl.col(h) if h in time_cols_in_template else pl.lit(None))
                  .alias(h)
            )
        elif h not in final.columns:
            final = final.with_columns(pl.lit(None).alias(h))

    # 보조 키/접미사 제거
    final = final.drop([c for c in final.columns if c.endswith("_pred")] + ["구간ID","방향_id"])

    # 최종 컬럼 순서 고정
    keep_left = [c for c in schema_left if c in final.columns]
    final = final.select(keep_left + [c for c in schema_right if c in final.columns])
    cols = [h for h in schema_right if h in final.columns]
    final = final.with_columns([
        pl.col(h).round(0)
                 .cast(pl.UInt8, strict=False)
                 .alias(h)
        for h in cols
    ])
    return final

In [ ]:
pred_longs = []
keys_sorted = sorted(list(pred_dict.keys()))
for key in keys_sorted:
    grid = pred_dict[key]
    pred_longs.append(to_long_polars(key, grid, segments))
pred_long_df = pl.concat(pred_longs, how="vertical", rechunk=True)

# === 9) 원본 스키마 검출 & 템플릿에 덮어쓰기 ===
schema_left, schema_right = get_schema(template)

pred_sheets = {}
for sheet_name, lf in template.items():
    # 시트명 → (월, 요일타입) 파싱
    m = int(re.search(r"^\d+", sheet_name).group())
    daytype = "평일" if "평일" in sheet_name else "주말·공휴일"
    d_id = DAY2ID[daytype]

    # 해당 시트용 예측 추출
    long_for_sheet = pred_long_df.filter((pl.col("월")==m) & (pl.col("요일타입_id")==d_id))

    # 템플릿에 예측 덮어쓰기
    final_df = to_polars(lf, schema_left=schema_left, schema_right=schema_right,
                                     pred_long_for_sheet=long_for_sheet)
    pred_sheets[sheet_name] = final_df

In [ ]:
import xlsxwriter

pred_xlsx = '예측 데이터.xlsx'
workbook = xlsxwriter.Workbook(pred_xlsx)

for k, v in pred_sheets.items():
  v.write_excel(workbook=workbook, worksheet=k, autofit=True)

workbook.close()